⚽ Agente Inteligente para Priorização de Alvos de Contratação
---------------------------------------------------------------------

**Aluno:** Matheus Sousa Marinho   
**Matrícula:** 202206132   
**Disciplina:** LIA 1 (2026/2)

Desenvolver um agente inteligente capaz de analisar uma base de atletas
e gerar automaticamente uma lista priorizada, identificando os jogadores
com maior potencial de reforço para um clube da Série A. A priorização é
realizada por meio de inferência com modelos de IA, considerando critérios
de scouting. O projeto utiliza modelos de linguagem (LLMs) para avaliar
cada atleta com base em múltiplos critérios, aplicando uma lógica de decisão
ponderada (pesos explícitos), simulando o raciocínio de um analista de futebol.

---------------------------------------------------------------------

📌 **Mesma estrutura do agente de priorização de leads, outro domínio**

*   Leads de ERP → atletas da Série A;
*   Potencial de compra → prioridade de contratação.

---------------------------------------------------------------------

🗂️ **Base de dados**

`atletas.csv`, com os 12 atletas que marcaram 10 gols ou mais na Série A 2024
e cinco critérios em faixas qualitativas. Gerado por `preparar_dados.py` a partir dos dados reais
do [Brasileirão Dataset](https://github.com/adaoduque/Brasileirao_Dataset).

---------------------------------------------------------------------

📊 **Saída Esperada**

score - atleta - justificativa - clube - posição

In [13]:
# Bibliotecas principais do projeto
import json
import re
import pandas as pd

In [14]:
# ============================================================
# PROVEDOR: GEMINI
# ============================================================
import os

from google import genai
from dotenv import load_dotenv

load_dotenv()
api_key = os.environ["GEMINI_API_KEY"]

# O notebook da aula cria o modelo com o pacote antigo (google-generativeai) mas chama
# client.models.generate_content, que e do pacote novo, e o 'client' nunca e criado.
# Aqui o cliente do SDK atual (google-genai) existe, e a chamada original funciona.
client = genai.Client(api_key=api_key)
PROVIDER = "gemini"
# gemini-2.5-flash, do notebook da aula, nao esta mais disponivel para chaves novas:
# a API responde 404 e indica migrar. Este e o flash estavel mais recente.
model = "gemini-3.7-flash"

In [ ]:
# Carregar a base de dados ou dataset
df = pd.read_csv("atletas.csv")
df

,id_atleta,atleta,clube,posicao,volume_ofensivo,regularidade,dependencia_de_penalti,risco_disciplinar,dificuldade_negociacao
0,1,Alerrandro Barra Mansa Realino de Souza,Vitoria,Atacante,muito alto,regular,alta,baixo,media
1,2,Estêvão,Palmeiras,Meio-campo,muito alto,constante,baixa,baixo,alta
2,3,Hulk,Atletico-MG,Atacante,bom,regular,alta,medio,media
3,4,José Manuel Alberto López,Palmeiras,Atacante,bom,constante,nula,medio,alta
4,5,Lucas Moura,Sao Paulo,Meio-campo,bom,constante,baixa,baixo,alta
5,6,Luciano da Rocha Neves,Sao Paulo,Meio-campo,alto,constante,baixa,alto,alta
6,7,Pablo Vegetti,Vasco,Atacante,alto,constante,nula,medio,media
7,8,Pedro,Flamengo,Atacante,alto,regular,alta,baixo,alta
8,9,Raphael Veiga,Palmeiras,Meio-campo,alto,regular,alta,medio,alta
9,10,Rodrigo Garro,Corinthians,Meio-campo,bom,regular,baixa,medio,media


In [16]:
# Montar o prompt
def montar_prompt(dados_para_analise):
    prompt = f"""
Você é um analista de scouting de um clube da Série A montando o elenco para 2025.

Analise a base de atletas abaixo e atribua um score de prioridade de 0 a 100
para cada jogador, do maior potencial de reforço para o menor.

UTILIZE EXPLICITAMENTE OS SEGUINTES PESOS:
- Volume ofensivo: 35%
- Regularidade ao longo da temporada: 20%
- Independência de pênalti: 15%
- Risco disciplinar: 15%
- Facilidade de negociação: 15%

COMO LER OS CAMPOS:
- volume_ofensivo: todos os avaliados marcaram 10 gols ou mais, entao "bom" e o piso
  desta lista e nao um volume fraco; acima dele vem "alto" e "muito alto"
- regularidade: "constante" marcou nos quatro quartos do campeonato, "concentrada" só numa janela curta
- dependencia_de_penalti: "nula" não converteu nenhum pênalti, "alta" depende deles
- risco_disciplinar: "alto" acumulou muitos cartões, e vermelho pesa mais que amarelo
- dificuldade_negociacao: "alta" é clube grande e negociação cara, "baixa" é clube rebaixado e mais disposto a vender

REGRAS IMPORTANTES:
- Avalie cada critério individualmente
- Combine os critérios de forma ponderada
- Pontue TODOS os atletas da base, sem exceção
- Retorne SOMENTE JSON válido
- Não use markdown
- Não escreva explicações antes ou depois
- A saída deve começar com {{ e terminar com }}

Formato esperado:
{{
  "atletas": [
    {{
      "id_atleta": 1,
      "atleta": "Nome do atleta",
      "score": 95,
      "justificativa": "Justificativa curta e objetiva"
    }}
  ]
}}

Base de atletas:
{json.dumps(dados_para_analise, ensure_ascii=False)}
"""
    return prompt

In [17]:
def gerar_resposta(prompt, provider, model):
    """
    Envia o prompt para o provedor escolhido e retorna o texto da resposta.
    """

    if provider == "gemini":
        response = client.models.generate_content(
            model=model,
            contents=prompt,
            config={
                "temperature": 0.2,
                "max_output_tokens": 32768,
                # Sem ferramentas no prompt; desligar evita um aviso do SDK.
                "automatic_function_calling": {"disable": True},
            },
        )
        return response.text

    else:
        raise ValueError("Provider inválido. Este notebook usa 'gemini'.")

In [18]:
def extrair_json(texto):
    """
    Tenta converter a resposta do modelo em JSON válido.
    Também trata casos em que o modelo retorna blocos markdown ou texto extra.
    """

    if not texto or not texto.strip():
        raise ValueError("O modelo retornou uma resposta vazia.")

    texto = texto.strip()

    # Remove blocos markdown, se existirem
    if texto.startswith("```json"):
        texto = texto.replace("```json", "", 1).strip()
    if texto.startswith("```"):
        texto = texto.replace("```", "", 1).strip()
    if texto.endswith("```"):
        texto = texto[:-3].strip()

    # Tenta converter diretamente
    try:
        return json.loads(texto)
    except json.JSONDecodeError:
        # Tenta extrair apenas o trecho JSON da resposta
        match = re.search(r"\{.*\}", texto, re.DOTALL)
        if match:
            return json.loads(match.group(0))
        raise ValueError(f"Não foi possível extrair um JSON válido.\n\nResposta recebida:\n{texto}")

In [19]:
# Preparar os dados para análise
dados_para_analise = df.fillna("").to_dict(orient="records")

# Montar o prompt
prompt = montar_prompt(dados_para_analise)

# Gerar resposta do modelo
conteudo = gerar_resposta(prompt, PROVIDER, model)

# Exibir resposta bruta para conferência
print("Resposta bruta do modelo:\n")
print(conteudo[:1500])

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [ ]:
# Converter resposta em JSON
resultado = extrair_json(conteudo)

# Transformar a lista de atletas em DataFrame
df_scores = pd.DataFrame(resultado["atletas"])

print(f"{len(df_scores)} atletas pontuados de {len(df)} enviados")
df_scores.head()

12 atletas pontuados de 12 enviados


,id_atleta,atleta,score,justificativa
0,2,Estêvão,86,"Volume muito alto, regularidade constante, bai..."
1,7,Pablo Vegetti,83,Alto volume com total independência de pênalti...
2,12,Yuri Alberto,81,Volume ofensivo máximo e alta regularidade com...
3,1,Alerrandro Barra Mansa Realino de Souza,78,"Excelente volume ofensivo, disciplina exemplar..."
4,5,Lucas Moura,72,Atleta constante e disciplinado com baixa depe...


In [ ]:
# Unir os scores gerados pela IA com dados originais da base
df_saida = df_scores.drop(columns=["atleta"]).merge(
    df[["id_atleta", "atleta", "clube", "posicao"]],
    on="id_atleta",
    how="left"
)

# Ordenar do maior score para o menor
df_saida = df_saida.sort_values(by="score", ascending=False).reset_index(drop=True)
df_saida.index += 1

# Visualizar resultado final
df_saida[["atleta", "clube", "posicao", "score", "justificativa"]].head(20)

,atleta,clube,posicao,score,justificativa
1,Estêvão,Palmeiras,Meio-campo,86,"Volume muito alto, regularidade constante, bai..."
2,Pablo Vegetti,Vasco,Atacante,83,Alto volume com total independência de pênalti...
3,Yuri Alberto,Corinthians,Atacante,81,Volume ofensivo máximo e alta regularidade com...
4,Alerrandro Barra Mansa Realino de Souza,Vitoria,Atacante,78,"Excelente volume ofensivo, disciplina exemplar..."
5,Lucas Moura,Sao Paulo,Meio-campo,72,Atleta constante e disciplinado com baixa depe...
6,Wesley Ribeiro Silva,Internacional,Meio-campo,72,"Alto volume ofensivo, regularidade contínua e ..."
7,José Manuel Alberto López,Palmeiras,Atacante,70,Totalmente independente de penalidades e const...
8,Luciano da Rocha Neves,Sao Paulo,Meio-campo,68,"Bom volume e constância de gols, mas fortement..."
9,Rodrigo Garro,Corinthians,Meio-campo,66,Baixa dependência de pênaltis e negociação int...
10,Pedro,Flamengo,Atacante,65,"Alto poder ofensivo e disciplina positiva, mas..."


In [ ]:
# Salvar o ranking
df_saida.to_csv("ranking_scouting_2024.csv", index=False, encoding="utf-8")
print("Ranking salvo em ranking_scouting_2024.csv")

Ranking salvo em ranking_scouting_2024.csv


### Observações

O agente entrega o ranking, mas vale saber o que ele não é. O score sai de uma
estimativa do modelo, não de uma conta: os pesos declarados no prompt são uma
instrução, e nada garante que foram aplicados na proporção exata. Rodar a mesma
célula de novo pode mudar a ordem.

Três limites da base:

1. Ela só enxerga quem fez gol ou levou cartão em 2024. Não há minutos jogados nem
   partidas disputadas, então um zagueiro regular não aparece.
2. `posicao` vem do arquivo de cartões, e quem nunca foi advertido fica sem ela.
3. Não há valor de mercado, salário, idade nem situação contratual, que são o que
   decide qualquer contratação de verdade.